<!-- NB_DOC_INTRO_V1 -->
# DL — PyTorch cheat-sheet

> 📚 **Doc thematique** : [docs/04_DL.md](docs/04_DL.md) (Deep Learning)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**PyTorch** = framework de DL de reference 2024-2025 en recherche, ~ a egalite avec TF pour la prod. API explicite (verbose mais clair), `autograd`, **eager execution** par defaut, compile pour la prod via `torch.compile` (>= 2.0).

Ce notebook execute un classifieur complet : `nn.Module`, `Dataset`/`DataLoader`, training loop manuel, `torch.compile`, save/load, MPS (Mac M-series) / CUDA.

Pour PyTorch Lightning (supprime le boilerplate) : [DL_PyTorch_Lightning.ipynb](./DL_PyTorch_Lightning.ipynb).

## Plan

1. Setup + devices (CPU / CUDA / MPS)
2. Tenseurs (creation, ops, autograd)
3. `nn.Module` (definir un modele)
4. `Dataset` + `DataLoader`
5. Training loop (forward, loss, backward, step)
6. Validation loop
7. `torch.compile` (PyTorch 2.0+)
8. Save/load (`state_dict`)
9. Pieges + References


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
import numpy as np
import warnings
warnings.filterwarnings("ignore")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"torch : {torch.__version__}")
print(f"CUDA : {torch.cuda.is_available()}")
if hasattr(torch.backends, "mps"):
    print(f"MPS  : {torch.backends.mps.is_available()}")

# Device auto
device = torch.device("cuda" if torch.cuda.is_available()
                       else "mps" if torch.backends.mps.is_available()
                       else "cpu")
print(f"Device choisi : {device}")

## 1. Tenseurs — creation, ops, autograd

In [ ]:
# Creation
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.zeros(3, 4)
c = torch.randn(3, 4)
d = torch.from_numpy(np.array([1, 2, 3]))  # interop numpy

print(f"a : {a}, shape={a.shape}, dtype={a.dtype}")
print(f"c shape {c.shape}, c.mean()={c.mean():.3f}")

# Move to device
a_dev = a.to(device)
print(f"a sur {a_dev.device}")

In [ ]:
# Autograd : differentiation automatique
x = torch.tensor([2.0], requires_grad=True)
y = x ** 3 + 2 * x + 1
y.backward()  # compute dy/dx
print(f"x = 2 ; y = x^3 + 2x + 1 = {y.item()}")
print(f"dy/dx = 3x^2 + 2 = {x.grad.item()}  (attendu : 14)")

## 2. `nn.Module` — definir un modele

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, in_dim=20, hidden=64, n_classes=2):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, n_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        return self.fc3(x)         # logits (pas de softmax — cross_entropy gere)

model = SimpleMLP(in_dim=20, hidden=64, n_classes=2).to(device)
print(model)
print(f"\nNb params : {sum(p.numel() for p in model.parameters()):,}")

## 3. `Dataset` + `DataLoader`

In [ ]:
# Donnees jouet (binaire)
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=2000, n_features=20, n_informative=15,
                            n_redundant=2, random_state=SEED)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)

# Convert numpy → torch
X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
y_tr_t = torch.tensor(y_tr, dtype=torch.long)
X_te_t = torch.tensor(X_te, dtype=torch.float32)
y_te_t = torch.tensor(y_te, dtype=torch.long)

# Wrap dans TensorDataset
train_ds = TensorDataset(X_tr_t, y_tr_t)
test_ds = TensorDataset(X_te_t, y_te_t)

# DataLoader (batch + shuffle + parallel)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=128, num_workers=0)
print(f"Train : {len(train_ds)} samples → {len(train_loader)} batches")

### Custom Dataset (pattern recommande pour datasets reels)

In [ ]:
class MyDataset(Dataset):
    """Pattern : tu peux y mettre du chargement lazy (fichiers images, lignes DB...)."""
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        if self.transform:
            x = self.transform(x)
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

# Test
ds = MyDataset(X_tr, y_tr)
print(f"len = {len(ds)}, item 0 = (shape {ds[0][0].shape}, label {ds[0][1].item()})")

## 4. Training loop manuel

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()       # important !
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()             # autograd
        optimizer.step()

        total_loss += loss.item() * X.size(0)
        correct += (logits.argmax(-1) == y).sum().item()
        total += y.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = criterion(logits, y)
        total_loss += loss.item() * X.size(0)
        correct += (logits.argmax(-1) == y).sum().item()
        total += y.size(0)
    return total_loss / total, correct / total

# Setup
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()

# Train 5 epochs
for epoch in range(1, 6):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    te_loss, te_acc = evaluate(model, test_loader, criterion, device)
    print(f"Epoch {epoch} : train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | "
          f"test_loss={te_loss:.4f} test_acc={te_acc:.4f}")

## 5. `torch.compile` (PyTorch 2.0+)

```python
compiled_model = torch.compile(model)   # speedup 1.3-2x sur GPU recent
# Puis exactement le meme code d'entrainement
```

Note : la 1ere iteration est lente (compilation), les suivantes sont rapides. Pas toujours utile sur CPU ou petits modeles.

## 6. Save / Load — `state_dict` (recommande)

In [ ]:
import tempfile, os
path = os.path.join(tempfile.gettempdir(), "model.pth")

# Save seulement les weights (recommande — independant de l'archi exact du code)
torch.save(model.state_dict(), path)
print(f"Sauve : {path}")

# Load
model_new = SimpleMLP(in_dim=20, hidden=64, n_classes=2).to(device)
model_new.load_state_dict(torch.load(path, map_location=device))
model_new.eval()

# Verifier
_, acc_after = evaluate(model_new, test_loader, criterion, device)
print(f"Acc apres reload : {acc_after:.4f}")

## 7. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| Oublier `optimizer.zero_grad()` | Gradients accumules → modele explose |
| Oublier `model.train()` / `model.eval()` | Dropout / BatchNorm comportement faux |
| Pas `@torch.no_grad()` en eval | Memoire gaspillee (autograd inutile) |
| `torch.save(model)` (whole) | Save `state_dict` → portable, smaller |
| Mix `.to(device)` partial | Tout (model + data) doit etre sur meme device |
| Softmax dans forward + CrossEntropyLoss | CrossEntropyLoss applique softmax — double softmax = bug |
| `DataLoader(num_workers=4)` sur Windows sans `if __name__ == '__main__'` | Crash multiprocessing |
| Pas de `weight_decay` | Souvent benefique (~ 1e-5) |
| `lr=1e-3` partout sans tester | Faire un LR finder (Smith 2017) |
| Pas de `torch.manual_seed` | Reproductibilite cassee |


## References

### Documentation
- PyTorch docs : https://pytorch.org/docs/stable/
- PyTorch tutorials : https://pytorch.org/tutorials/
- torch.compile : https://pytorch.org/docs/stable/torch.compiler.html

### Voir aussi
- [DL_Deep_Learning_Maths.ipynb](DL_Deep_Learning_Maths.ipynb)
- [DL_Tensorflow_Keras.ipynb](DL_Tensorflow_Keras.ipynb)
- [DL_PyTorch_Lightning.ipynb](DL_PyTorch_Lightning.ipynb)


<!-- ENRICH_2025_V1 -->

## 🔥 PyTorch 2024-2025 — techniques modernes

### 1. **`torch.compile`** (Pytorch 2.0+)

Compile le graphe d'execution en C++/CUDA → **1.3-2× speedup** sur GPU recent.

```python
model = torch.compile(model)   # Wrapping
# Puis exactement le meme training loop
```

Premiere iteration lente (compilation), mais ensuite tres rapide. Mode `mode="reduce-overhead"` pour inference.

### 2. **`torch.set_float32_matmul_precision`**

Sur Ampere+ (RTX 30/40 series, A100, H100), permet d'utiliser **TF32** (vs FP32). Gain 2-4× speedup avec perte negligeable.

```python
torch.set_float32_matmul_precision("high")   # ou "medium"
```

### 3. **Flash Attention 2/3**

Attention exact mais 2-4× plus rapide + memoire reduite. Standard pour LLMs.

```python
# Active automatiquement dans transformers >= 4.36 si torch >= 2.1 + GPU compatible
from transformers import AutoModel
model = AutoModel.from_pretrained("...", attn_implementation="flash_attention_2")
```

### 4. **`torch.bfloat16`** > `float16`

BF16 a la **meme range** que FP32 (juste moins de precision dans la mantisse). Plus stable pour training, moins de NaN. Standard sur Ampere+.

```python
model = model.to(torch.bfloat16)
# Ou en training : autocast(dtype=torch.bfloat16)
```

### 5. **`torch.utils.checkpoint`** — gradient checkpointing

Trade memoire contre temps : recompute les activations forward au lieu de les stocker. Gain memoire 2-3×, temps +20%.

```python
from torch.utils.checkpoint import checkpoint

class MyLayer(nn.Module):
    def forward(self, x):
        return checkpoint(self._forward, x, use_reentrant=False)
```

### 6. **FSDP (Fully Sharded Data Parallel)**

Shard les **parametres**, **gradients** et **optimizer states** sur plusieurs GPUs. Permet de fit un Llama 70B sur 8×A100.

```python
from torch.distributed.fsdp import FullyShardedDataParallel
model = FullyShardedDataParallel(model, ...)
```

DeepSpeed ZeRO-3 = equivalent + plus mature.

### 7. **`torch.cuda.amp.autocast`** + `GradScaler` — Mixed Precision

```python
from torch.amp import autocast, GradScaler
scaler = GradScaler()
for x, y in loader:
    optimizer.zero_grad()
    with autocast(device_type="cuda", dtype=torch.bfloat16):
        out = model(x)
        loss = criterion(out, y)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
```

### 8. **`@torch.no_grad`** + `@torch.inference_mode`

Pour l'inference, `inference_mode()` est encore plus rapide que `no_grad()` (Pytorch 1.9+).

```python
@torch.inference_mode()
def predict(model, x):
    return model(x)
```

### 9. **`nn.utils.parametrizations.spectral_norm`** — stabilite GAN

Normalisation spectrale, plus stable que weight clipping.

### 10. **Quantization API** (`torch.ao.quantization`)

Pour deployer en INT8 / FP16 :
```python
from torch.ao.quantization import quantize_dynamic
quantized = quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)
# Modele 4× plus petit, ~ 2× plus rapide en inference CPU
```

## 🔥 Alternatives a PyTorch en 2024-2025

| Framework | Forces |
|---|---|
| **JAX** (Google) | Pure functions, JIT, TPU-native — recherche math |
| **MLX** (Apple) | Mac M-series natif, API NumPy-like |
| **TinyGrad** (G. Hotz) | Mini-framework educatif (~ 5k lignes) |
| **MicroGrad** (Karpathy) | Encore plus mini, pedagogique |
| **Tinytensor** | Production tinyengine |

PyTorch reste **dominant** (~ 80% recherche + prod) en 2025.

## 💡 Checklist deployment PyTorch 2025

- [ ] `torch.compile(model)` pour speedup gratuit
- [ ] `bfloat16` plutot que `float16` (plus stable)
- [ ] `flash_attention_2` si Transformer
- [ ] `torch.inference_mode()` pour inference
- [ ] Export en **ONNX** ou **TorchScript** pour prod sans Python
- [ ] **`torch.export`** (Pytorch 2.1+) pour export propre
- [ ] **vLLM** ou **TGI** pour servir des LLMs (vs `model.generate` naif)
- [ ] **Quantization** INT8 / FP8 pour deploy edge
